In [ ]:
!pip uninstall torchvision -y
!pip install whisperx
!pip install pydub

In [ ]:
import whisperx
import gc
import torch
from pydub import AudioSegment, effects
from google.colab import files, userdata
from whisperx.diarize import DiarizationPipeline

# --- 1. Загрузка файла через диалог ---
uploaded = files.upload()
if not uploaded:
    raise ValueError("Файл не был выбран")

filename = list(uploaded.keys())[0]
print(f"Обработка файла: {filename}")

# --- 2. Нормализация (AGC) ---
# Приводим звук к единому стандарту громкости
raw_audio = AudioSegment.from_file(filename)
normalized_audio = effects.normalize(raw_audio)
temp_audio_path = "normalized_audio.wav"
normalized_audio.export(temp_audio_path, format="wav")

# --- 3. Настройка параметров и моделей ---
device = "cuda"
compute_type = "float16"
HF_TOKEN = userdata.get("HF_TOKEN")

# Загружаем модель с чуть более строгим VAD (порог 0.5), чтобы отсечь шумы
model = whisperx.load_model(
    "large-v2", 
    device, 
    compute_type=compute_type,
    vad_options={"vad_onset": 0.5, "vad_offset": 0.363}
)

# Загружаем нормализованный звук
audio = whisperx.load_audio(temp_audio_path)

# --- 4. Транскрипция и выравнивание ---
result = model.transcribe(audio, batch_size=16)

model_a, metadata = whisperx.load_align_model(
    language_code=result["language"], device=device
)
result = whisperx.align(
    result["segments"], model_a, metadata, audio, device
)

# Очистка памяти для диаризации
del model_a
gc.collect()
torch.cuda.empty_cache()

# --- 5. Диаризация (Распознавание игроков) ---
diarize_model = DiarizationPipeline(token=HF_TOKEN, device=device)

# min_speakers=1, max_speakers=5 (подставь свой максимум)
diarize_segments = diarize_model(audio, min_speakers=5, max_speakers=15)
result = whisperx.assign_word_speakers(diarize_segments, result)

# --- 6. Вывод результата в консоль ---
print("\n" + "="*30)
print("РЕЗУЛЬТАТ РАСПОЗНАВАНИЯ")
print("="*30 + "\n")

for seg in result["segments"]:
    speaker = seg.get("speaker", "UNKNOWN")
    text = seg['text'].strip()
    if text:
        print(f"[{speaker}] [{seg['start']:.1f}s → {seg['end']:.1f}s]: {text}")

# Финальная очистка ресурсов
del model
gc.collect()
torch.cuda.empty_cache()